In [ ]:
-- 6.3 Запрос для подсчета метрик по взаимодействиям с пуш-уведомлениями и оформлением подписок

with push_base as (

    -- 1) Получение даты пуш-уведомлений клиентам
    select
        contact_id,
        to_date(substr(campaign_name, 1, 8), 'YYYYMMDD') as push_dt,
        is_sent::int as is_sent,
        is_delivered::int as is_delivered,
        is_opened::int as is_opened
    from cvm_sbx.{prefix}_CVMB24118_push

),

subscr_min as (

    -- 2) Определение даты первого оформления подписки
    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_date
    from cvm_sbx.{prefix}_CVMB24118_subscr_status
    group by contact_id

),

push_client as (

    -- 3) Агрегация флагов по пушам клиентов
    select
        pb.contact_id,

        max(pb.is_sent) as is_sent,
        max(pb.is_delivered) as is_delivered,
        max(pb.is_opened) as is_opened,

        max(case
            when pb.is_opened = 1
             and (
                    sm.first_subscr_date is null
                    or sm.first_subscr_date < pb.push_dt
                 )
            then 1
            else 0
        end) as opened_without_subscr

    from push_base pb

    left join subscr_min sm
        on pb.contact_id = sm.contact_id

    group by pb.contact_id

),

base as (

    -- 4) Объединение с когортами клиентов
    select
        cc.client_id,
        cc.campaigns_cnt,

        coalesce(pc.is_sent, 0) as is_sent,
        coalesce(pc.is_delivered, 0) as is_delivered,
        coalesce(pc.is_opened, 0) as is_opened,
        coalesce(pc.opened_without_subscr, 0) as opened_without_subscr

    from cvm_sbx.{prefix}_CVMB_24118_client_cohorts cc

    left join push_client pc
        on cc.client_id = pc.contact_id

),

cohort_result as (

    -- 5) Подсчет абсолютных значений по когортам
    select
        campaigns_cnt,

        count(distinct client_id) as total_clients,

        count(distinct case
            when is_sent = 1 then client_id
        end) as sent_clients,

        count(distinct case
            when is_delivered = 1 then client_id
        end) as delivered_clients,

        count(distinct case
            when is_opened = 1 then client_id
        end) as opened_clients,

        count(distinct case
            when opened_without_subscr = 1 then client_id
        end) as opened_without_subscr_clients

    from base

    group by campaigns_cnt

)

-- 6) Расчет долей по когортам
select
    campaigns_cnt,

    total_clients,

    sent_clients,
    delivered_clients,
    opened_clients,
    opened_without_subscr_clients,

    round(sent_clients * 100.0 / nullif(total_clients, 0), 1) as sent_pct,

    round(delivered_clients * 100.0 / nullif(total_clients, 0), 1) as delivered_pct,

    round(opened_clients * 100.0 / nullif(total_clients, 0), 1) as opened_pct,

    round(
        opened_without_subscr_clients * 100.0
        / nullif(opened_clients, 0),
        1
    ) as opened_without_subscr_pct_from_opened,

    round(
        opened_without_subscr_clients * 100.0
        / nullif(total_clients, 0),
        1
    ) as opened_without_subscr_pct_total

from cohort_result

order by campaigns_cnt;